# Structured Campaign Tutorial

This notebook demonstrates a lightweight staged campaign with two explicit stages: `screening` and `refinement`. The screening stage uses a smaller active variable set, while refinement activates temperature and residence time after early chemistry has been established.

The CSV log remains the source of truth. Suggestions are dry runs until they are explicitly appended.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

mpl_cache = PROJECT_ROOT / ".matplotlib-cache"
mpl_cache.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from bo_forge import CampaignSession

CONFIG_PATH = PROJECT_ROOT / "configs" / "14_structured_campaign_tutorial.yaml"
SEED_LOG_PATH = PROJECT_ROOT / "examples" / "14_structured_campaign_tutorial_campaign_log.csv"
WORKING_LOG_PATH = PROJECT_ROOT / "examples" / "14_structured_campaign_tutorial_working_log.csv"
LATEST_SUGGESTIONS_PATH = PROJECT_ROOT / "examples" / "14_structured_campaign_tutorial_latest_suggestions.csv"
REPORT_PATH = PROJECT_ROOT / "reports" / "14_structured_campaign_tutorial_report.txt"
STAGE_DIAGNOSTICS_PATH = PROJECT_ROOT / "reports" / "14_structured_campaign_tutorial_stage_diagnostics.png"
TARGET_OBSERVED_ROWS = 15

shutil.copyfile(SEED_LOG_PATH, WORKING_LOG_PATH)
campaign = CampaignSession.from_files(CONFIG_PATH, WORKING_LOG_PATH)

## Validate and inspect the staged state

`summary()` gives the campaign-level view. `stage_summary()` shows row counts, active variables, inactive variables, and read-only transition-readiness guidance for each configured stage.

In [ ]:
campaign.validate()
campaign.summary()

In [ ]:
campaign.stage_summary()

## Dry-run a screening suggestion

Structured campaigns with multiple stages require an explicit stage. This call is non-mutating: it returns a staged suggestion table but does not write to the CSV log.

In [ ]:
screening_suggestions = campaign.suggest_next(batch_size=1, stage="screening")
screening_suggestions.to_csv(LATEST_SUGGESTIONS_PATH, index=False)
screening_suggestions

## Append explicitly and simulate an observation

The synthetic objective is deterministic and cheap. In a real campaign this step would be replaced by the experimental result.

In [ ]:
def synthetic_activity(row):
    loading = float(row["catalyst_loading"])
    base_bonus = 0.12 if row["base"] == "K2CO3" else 0.0
    temperature = row["temperature"]
    residence_time = row["residence_time"]
    if temperature == "" or residence_time == "":
        return 0.85 + 1.30 * loading + base_bonus - 0.40 * (loading - 0.28) ** 2
    return (
        1.00
        + 1.40 * loading
        + base_bonus
        + 0.003 * (float(temperature) - 65.0)
        + 0.002 * float(residence_time)
        - 0.50 * (loading - 0.32) ** 2
    )


campaign.append_suggestions(screening_suggestions)
for row_id in screening_suggestions["row_id"]:
    row = campaign.df.loc[campaign.df["row_id"] == row_id].iloc[0]
    campaign.mark_observed(row_id=row_id, objective_value=synthetic_activity(row))
campaign.reload()
campaign.stage_summary()

## Refine with the larger active variable set

The refinement stage activates `temperature` and `residence_time`. Stage-aware constraints are evaluated only when all variables referenced by the constraint are active in the selected stage.

In [ ]:
refinement_suggestions = campaign.suggest_next(batch_size=1, stage="refinement")
refinement_suggestions.to_csv(LATEST_SUGGESTIONS_PATH, index=False)
campaign.append_suggestions(refinement_suggestions)
for row_id in refinement_suggestions["row_id"]:
    row = campaign.df.loc[campaign.df["row_id"] == row_id].iloc[0]
    campaign.mark_observed(row_id=row_id, objective_value=synthetic_activity(row))
campaign.reload()
campaign.stage_summary()

## Report and diagnostics

Reports include a structured stage section. Stage diagnostics show row counts by stage and an active/inactive variable map.

In [ ]:
campaign.export_report(REPORT_PATH)
_fig, _axes = campaign.plot_stage_diagnostics(save_path=STAGE_DIAGNOSTICS_PATH)
REPORT_PATH, STAGE_DIAGNOSTICS_PATH

## Compact CLI demo

The same backend behavior is available through `python -m bo_forge`.

In [ ]:
completed = subprocess.run(
    [
        sys.executable,
        "-m",
        "bo_forge",
        "stage-summary",
        "--config",
        str(CONFIG_PATH),
        "--log",
        str(WORKING_LOG_PATH),
    ],
    check=True,
    capture_output=True,
    cwd=PROJECT_ROOT,
    text=True,
)
print(completed.stdout)

## Limitations

- Stage selection is explicit; BO Forge does not automatically transition between stages.
- This is not multi-fidelity or contextual BO.
- Streamlit structured workflow polish remains out of scope for this tutorial.
- Cost-aware structured campaigns remain deferred.
- The notebook demonstrates backend/session/CLI behavior; it is not the campaign engine.